# API Data Transformation — Chama

Flatten event payloads from `case.json` into three analytics-ready CSV tables using Python, Pandas, and JSON parsing.

## Table of Contents

1. [Assignment Summary](#assignment-summary)
2. [Approach](#approach)
3. [Load Libraries and Data](#load-libraries-and-data)
4. [Initial Data Exploration](#initial-data-exploration)
5. [Timestamp Conversion](#timestamp-conversion)
6. [Helper Functions](#helper-functions)
7. [DynamicPriceOption.csv](#dynamicpriceoptioncsv)
8. [DynamicPriceRange.csv](#dynamicpricerangecsv)
9. [CuratedOfferOptions.csv](#curatedofferoptionscsv)
10. [Export CSV Files](#export-csv-files)
11. [Validation and Output Preview](#validation-and-output-preview)

## Assignment Summary

Chama receives application and backend events as JSON records. Each record has:

- `EnqueuedTimeUtc`: event timestamp in UTC;
- `EventName`: event type;
- `Payload`: event-specific JSON content stored as a string.

The goal is to transform `case.json` into three CSV files:

### `CuratedOfferOptions.csv`

| Column | Required format |
|---|---|
| `CurationProvider` | in quotes |
| `OfferId` | in quotes |
| `DealerId` | in quotes |
| `UniqueOptionId` | in quotes |
| `OptionId` | in quotes |
| `IsMobileDealer` | without quotes |
| `IsOpen` | without quotes |
| `Eta` | in quotes |
| `ChamaScore` | without quotes |
| `ProductBrand` | in quotes |
| `IsWinner` | without quotes |
| `MinimumPrice` | without quotes |
| `MaximumPrice` | without quotes |
| `DynamicPrice` | without quotes |
| `FinalPrice` | without quotes |
| `DefeatPrimaryReason` | in quotes |
| `DefeatReasons` | in quotes |
| `EnqueuedTimeSP` | `DD/MM/YYYY`, converted to UTC-3 |

### `DynamicPriceOption.csv`

| Column | Required format |
|---|---|
| `Provider` | in quotes |
| `OfferId` | in quotes |
| `UniqueOptionId` | in quotes |
| `BestPrice` | without quotes |
| `EnqueuedTimeSP` | `DD/MM/YYYY`, converted to UTC-3 |

### `DynamicPriceRange.csv`

| Column | Required format |
|---|---|
| `Provider` | in quotes |
| `OfferId` | in quotes |
| `MinGlobal` | without quotes |
| `MinRecommended` | without quotes |
| `MaxRecommended` | without quotes |
| `DifferenceMinRecommendMinTheory` | without quotes |
| `EnqueuedTimeSP` | `DD/MM/YYYY`, converted to UTC-3 |

## Approach

The transformation is split into small, testable steps:

1. Load `case.json` into a DataFrame.
2. Convert `EnqueuedTimeUtc` to `EnqueuedTimeSP` using UTC-3 and format it as `DD/MM/YYYY`.
3. Split the data by `EventName`.
4. Parse the nested `Payload` JSON for each event type.
5. Flatten each payload into one row per required output record.
6. Export the three CSV files with the exact required names and quote rules.

## Load Libraries and Data

In [1]:
import json
import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_json('case.json')
df.head()

,EnqueuedTimeUtc,EventName,Payload
0,2021-09-05 08:04:08 UTC,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."
1,2021-08-18 11:43:23 UTC,DynamicPrice_Result,"{""provider"":""ApplyDynamicPricePerOption"",""offe..."
2,2021-09-05 09:04:04 UTC,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."
3,2021-08-25 05:02:55 UTC,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""149f..."
4,2021-09-05 08:03:28 UTC,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."


## Initial Data Exploration

Before flattening the JSON payloads, it is useful to confirm the size of the file, the available columns, and the distribution of event types.

In [2]:
df.shape

(37, 3)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   EnqueuedTimeUtc  37 non-null     object
 1   EventName        37 non-null     object
 2   Payload          37 non-null     object
dtypes: object(3)
memory usage: 1020.0+ bytes


In [4]:
df["EventName"].value_counts()

,count
EventName,
DynamicPrice_Result,32
CurateOffer_Result,5


In [5]:
df.sample(n=5, random_state=42)

,EnqueuedTimeUtc,EventName,Payload
17,2021-08-25 09:02:29 UTC,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""0a06..."
13,2021-08-25 09:03:29 UTC,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""c99a..."
4,2021-09-05 08:03:28 UTC,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."
29,2021-09-05 09:04:17 UTC,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."
35,2021-08-25 09:03:14 UTC,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""135d..."


There are two event names in the file:

- `DynamicPrice_Result`: used to create `DynamicPriceOption.csv` and `DynamicPriceRange.csv`.
- `CurateOffer_Result`: used to create `CuratedOfferOptions.csv`.

The `Payload` column is a JSON string, so each payload must be parsed before its nested fields can be extracted.

## Timestamp Conversion

The assignment asks for `EnqueuedTimeSP` as `DD/MM/YYYY`, converted from UTC to Brazilian timezone UTC-3. Since the requirement explicitly says UTC-3, the conversion below subtracts three hours from the UTC timestamp and then formats the date.

In [6]:
df["EnqueuedTimeSP"] = (
    pd.to_datetime(df["EnqueuedTimeUtc"], utc=True)
    - pd.Timedelta(hours=3)
).dt.strftime("%d/%m/%Y")

df[["EnqueuedTimeUtc", "EnqueuedTimeSP", "EventName"]].head()

,EnqueuedTimeUtc,EnqueuedTimeSP,EventName
0,2021-09-05 08:04:08 UTC,05/09/2021,DynamicPrice_Result
1,2021-08-18 11:43:23 UTC,18/08/2021,DynamicPrice_Result
2,2021-09-05 09:04:04 UTC,05/09/2021,DynamicPrice_Result
3,2021-08-25 05:02:55 UTC,25/08/2021,CurateOffer_Result
4,2021-09-05 08:03:28 UTC,05/09/2021,DynamicPrice_Result


## Helper Functions

The helper functions below keep the solution organized:

- `parse_payload`: converts the payload string into a Python dictionary or list.
- `build_dynamic_price_option`: flattens per-option dynamic pricing payloads.
- `build_dynamic_price_range`: flattens dynamic pricing range payloads.
- `build_curated_offer_options`: flattens curated offer payloads, including nested dealer options.
- `write_csv_with_required_quotes`: writes a CSV while quoting only the columns required by the assignment.

In [24]:
DYNAMIC_PRICE_OPTION_COLUMNS = [
    "Provider",
    "OfferId",
    "UniqueOptionId",
    "BestPrice",
    "EnqueuedTimeSP",
]

DYNAMIC_PRICE_RANGE_COLUMNS = [
    "Provider",
    "OfferId",
    "MinGlobal",
    "MinRecommended",
    "MaxRecommended",
    "DifferenceMinRecommendMinTheory",
    "EnqueuedTimeSP",
]

CURATED_OFFER_OPTIONS_COLUMNS = [
    "CurationProvider",
    "OfferId",
    "DealerId",
    "UniqueOptionId",
    "OptionId",
    "IsMobileDealer",
    "IsOpen",
    "Eta",
    "ChamaScore",
    "ProductBrand",
    "IsWinner",
    "MinimumPrice",
    "MaximumPrice",
    "DynamicPrice",
    "FinalPrice",
    "DefeatPrimaryReason",
    "DefeatReasons",
    "EnqueuedTimeSP",
]


def parse_payload(payload: str):
    """Parse the JSON string stored in the Payload column."""
    return json.loads(payload)


def build_dynamic_price_option(events: pd.DataFrame) -> pd.DataFrame:
    """Create the DynamicPriceOption output table."""
    rows = []

    for _, row in events.iterrows():
        payload = parse_payload(row["Payload"])

        if payload.get("provider") != "ApplyDynamicPricePerOption":
            continue

        for option in payload.get("algorithmOutput", []):
            rows.append(
                {
                    "Provider": payload.get("provider"),
                    "OfferId": payload.get("offerId"),
                    "UniqueOptionId": option.get("uniqueOptionId"),
                    "BestPrice": option.get("bestPrice"),
                    "EnqueuedTimeSP": row["EnqueuedTimeSP"],
                }
            )

    return pd.DataFrame(rows, columns=DYNAMIC_PRICE_OPTION_COLUMNS)


def build_dynamic_price_range(events: pd.DataFrame) -> pd.DataFrame:
    """Create the DynamicPriceRange output table."""
    rows = []

    for _, row in events.iterrows():
        payload = parse_payload(row["Payload"])

        if payload.get("provider") != "ApplyDynamicPriceRange":
            continue

        algorithm_output = payload.get("algorithmOutput", {})

        rows.append(
            {
                "Provider": payload.get("provider"),
                "OfferId": payload.get("offerId"),
                "MinGlobal": algorithm_output.get("min_global"),
                "MinRecommended": algorithm_output.get("min_recommended"),
                "MaxRecommended": algorithm_output.get("max_recommended"),
                "DifferenceMinRecommendMinTheory": algorithm_output.get(
                    "differenceMinRecommendMinTheory"
                ),
                "EnqueuedTimeSP": row["EnqueuedTimeSP"],
            }
        )

    return pd.DataFrame(rows, columns=DYNAMIC_PRICE_RANGE_COLUMNS)


def build_curated_offer_options(events: pd.DataFrame) -> pd.DataFrame:
    """Create the CuratedOfferOptions output table."""
    rows = []

    for _, row in events.iterrows():
        payload = parse_payload(row["Payload"])

        for offer in payload:
            for option in offer.get("options", []):
                defeat_reasons = option.get("defeatReasons", "")
                if isinstance(defeat_reasons, list):
                    defeat_reasons = str(defeat_reasons)

                rows.append(
                    {
                        "CurationProvider": offer.get("curationProvider"),
                        "OfferId": offer.get("offerId"),
                        "DealerId": offer.get("dealerId"),
                        "UniqueOptionId": option.get("uniqueOptionId"),
                        "OptionId": option.get("optionId"),
                        "IsMobileDealer": option.get("isMobileDealer"),
                        "IsOpen": option.get("isOpen"),
                        "Eta": option.get("eta"),
                        "ChamaScore": option.get("chamaScore"),
                        "ProductBrand": option.get("productBrand"),
                        "IsWinner": option.get("isWinner"),
                        "MinimumPrice": option.get("minimumPrice"),
                        "MaximumPrice": option.get("maximumPrice"),
                        "DynamicPrice": option.get("dynamicPrice"),
                        "FinalPrice": option.get("finalPrice"),
                        "DefeatPrimaryReason": option.get("defeatPrimaryReason", ""),
                        "DefeatReasons": defeat_reasons,
                        "EnqueuedTimeSP": row["EnqueuedTimeSP"],
                    }
                )

    return pd.DataFrame(rows, columns=CURATED_OFFER_OPTIONS_COLUMNS)


def csv_scalar(value) -> str:
    """Convert a scalar value to its CSV text representation."""
    if pd.isna(value):
        return ""
    return str(value)


def write_csv_with_required_quotes(
    data: pd.DataFrame,
    output_path: str,
    quoted_columns: set[str],
) -> None:
    """
    Write a CSV while quoting only assignment-specified columns.

    Pandas' default CSV writer does not support quoting only selected columns.
    Because this assignment explicitly requires some columns to be quoted and
    others to remain unquoted, the output is written row by row.
    """

    with open(output_path, "w", encoding="utf-8", newline="") as file:
        file.write(",".join(data.columns) + "\n")

        for _, row in data.iterrows():
            values = []

            for column in data.columns:
                value = csv_scalar(row[column])

                if column in quoted_columns:
                    value = '"' + value.replace('"', '""') + '"'

                values.append(value)

            file.write(",".join(values) + "\n")

## `DynamicPriceOption.csv`

Rows for this output come from `DynamicPrice_Result` events where the payload provider is `ApplyDynamicPricePerOption`. Each item inside `algorithmOutput` becomes one row.

In [10]:
dynamic_price_events = df[df["EventName"] == "DynamicPrice_Result"].copy()
dynamic_price_events.shape

(32, 4)

In [11]:
dynamic_price_option = build_dynamic_price_option(dynamic_price_events)
dynamic_price_option.shape

(56, 5)

In [12]:
dynamic_price_option.head(10)

,Provider,OfferId,UniqueOptionId,BestPrice,EnqueuedTimeSP
0,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,b0e296a9-0590-f0e0-8211-243a2ededb12,92.45,18/08/2021
1,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,d6562c24-0b37-5fb4-8275-65b7b8b47b87,92.45,18/08/2021
2,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,8d0f9262-f543-d0c8-a869-33985ae3ecda,92.45,18/08/2021
3,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,151e59ac-761a-96f5-d2b9-882037a9fd28,94.60,18/08/2021
4,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,3cd346f4-d297-7568-2e50-d43a8e2fd0a9,94.60,18/08/2021
5,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,b7a7b6d1-4dae-7392-5aaf-f3369c29db1d,93.00,18/08/2021
6,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,577e4bbd-f49d-ac23-56a6-e70072a05229,93.00,18/08/2021
7,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,f9b876ab-2590-952f-d69d-5b352ec251f3,91.35,18/08/2021
8,ApplyDynamicPricePerOption,00991873-194e-4a6e-89c9-8f68668b6aaa,b0e296a9-0590-f0e0-8211-243a2ededb12,92.45,18/08/2021
9,ApplyDynamicPricePerOption,00991873-194e-4a6e-89c9-8f68668b6aaa,d6562c24-0b37-5fb4-8275-65b7b8b47b87,92.45,18/08/2021


## `DynamicPriceRange.csv`

Rows for this output come from `DynamicPrice_Result` events where the payload provider is `ApplyDynamicPriceRange`. In these events, `algorithmOutput` is a dictionary containing the range fields.

In [13]:
dynamic_price_range = build_dynamic_price_range(dynamic_price_events)
dynamic_price_range.shape

(25, 7)

In [14]:
dynamic_price_range.head(10)

,Provider,OfferId,MinGlobal,MinRecommended,MaxRecommended,DifferenceMinRecommendMinTheory,EnqueuedTimeSP
0,ApplyDynamicPriceRange,a6611d55-9624-4381-8cdd-323ee3689241,85.00,87.20,97.65,2.2,05/09/2021
1,ApplyDynamicPriceRange,b8c636fa-8241-47dc-ac40-bdf438a04d9c,85.00,87.20,97.65,2.2,05/09/2021
2,ApplyDynamicPriceRange,3d32f7fb-396d-4d3f-b673-dea1f7dc41b7,85.00,87.20,97.65,2.2,05/09/2021
3,ApplyDynamicPriceRange,329194f3-95a4-45ef-b3d0-2796f74ce2a0,85.00,87.20,97.65,2.2,05/09/2021
4,ApplyDynamicPriceRange,fdcfde5c-113d-4a59-9ae0-8bc31e2943d8,87.35,89.25,99.95,1.9,05/09/2021
5,ApplyDynamicPriceRange,27bbc4fa-2388-4780-b66c-92a51397d191,87.35,89.25,99.95,1.9,05/09/2021
6,ApplyDynamicPriceRange,baffc30b-7642-45fe-a2ce-da31a71732ae,85.00,87.20,97.65,2.2,05/09/2021
7,ApplyDynamicPriceRange,b5982abd-f602-47ac-b45a-bb43bf993d46,85.00,87.20,97.65,2.2,05/09/2021
8,ApplyDynamicPriceRange,f6643886-4a0f-45ae-ae32-ee95c72ee94a,87.35,89.25,99.95,1.9,05/09/2021
9,ApplyDynamicPriceRange,16a2d492-b1c3-40ec-970f-b8704d8db96f,85.00,87.20,97.65,2.2,05/09/2021


## `CuratedOfferOptions.csv`

Rows for this output come from `CurateOffer_Result` events. Each payload contains offers, each offer contains dealer information, and each dealer offer contains one or more options. Each nested option becomes one output row.

In [15]:
curated_offer_events = df[df["EventName"] == "CurateOffer_Result"].copy()
curated_offer_events.shape

(5, 4)

In [16]:
curated_offer_options = build_curated_offer_options(curated_offer_events)
curated_offer_options.shape

(40, 18)

In [17]:
curated_offer_options.head(10)

,CurationProvider,OfferId,DealerId,UniqueOptionId,OptionId,IsMobileDealer,IsOpen,Eta,ChamaScore,ProductBrand,IsWinner,MinimumPrice,MaximumPrice,DynamicPrice,FinalPrice,DefeatPrimaryReason,DefeatReasons,EnqueuedTimeSP
0,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,6517,b0e296a9-0590-f0e0-8211-243a2ededb12,6517 || dd839e4c-9f84-45eb-9cb2-9069fecf70f2,True,True,1:00,8.0,ULTRAGAZ,True,90.00,180.00,91.90,91.90,,,25/08/2021
1,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,6517,d6562c24-0b37-5fb4-8275-65b7b8b47b87,6517 || 6517,False,False,0:01,8.0,ULTRAGAZ,False,90.00,180.00,91.90,91.90,Closed,"['Closed', 'HasDriverInOffer']",25/08/2021
2,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,9047,8d0f9262-f543-d0c8-a869-33985ae3ecda,9047 || 9047 || ULTRAGAZ,False,False,1:00,9.0,ULTRAGAZ,False,99.00,198.00,99.95,99.95,Closed,"['Closed', 'HigherPrice', 'HasDriverInOffer']",25/08/2021
3,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,9047,3cd346f4-d297-7568-2e50-d43a8e2fd0a9,9047 || 9047 || CONSIGAZ,False,False,1:00,9.0,CONSIGAZ,False,89.99,179.98,91.89,91.89,Closed,"['Closed', 'HigherPrice', 'HigherETA']",25/08/2021
4,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,9047,577e4bbd-f49d-ac23-56a6-e70072a05229,9047 || 9047 || LIQUIGAS,False,False,1:00,9.0,LIQUIGAS,False,92.00,184.00,93.90,93.90,Closed,"['Closed', 'HigherPrice', 'HigherETA']",25/08/2021
5,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,9047,f9b876ab-2590-952f-d69d-5b352ec251f3,9047 || 9047 || NACIONALGAS,False,False,1:00,9.0,NACIONALGAS,True,88.90,177.80,90.80,90.80,,,25/08/2021
6,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,8330,151e59ac-761a-96f5-d2b9-882037a9fd28,8330 || 8330 || CONSIGAZ,False,True,0:01,8.0,CONSIGAZ,True,80.00,160.00,87.35,87.35,,,25/08/2021
7,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,8835,b7a7b6d1-4dae-7392-5aaf-f3369c29db1d,8835 || 8835 || LIQUIGAS,False,False,0:01,8.0,LIQUIGAS,True,60.00,120.00,87.35,87.35,,,25/08/2021
8,ByPrice,c99aa9a7-ac50-4a10-be0f-ac9f5ac0f45d,6517,b0e296a9-0590-f0e0-8211-243a2ededb12,6517 || dd839e4c-9f84-45eb-9cb2-9069fecf70f2,True,True,00:12:54.9215999,8.0,ULTRAGAZ,True,90.00,180.00,91.90,91.90,,,25/08/2021
9,ByPrice,c99aa9a7-ac50-4a10-be0f-ac9f5ac0f45d,6517,d6562c24-0b37-5fb4-8275-65b7b8b47b87,6517 || 6517,False,True,0:01,8.0,ULTRAGAZ,False,90.00,180.00,91.90,91.90,HasDriverInOffer,['HasDriverInOffer'],25/08/2021


## Export CSV Files

The output file names below match the assignment exactly.

In [25]:
dynamic_price_option_path = "DynamicPriceOption.csv"
dynamic_price_range_path = "DynamicPriceRange.csv"
curated_offer_options_path = "CuratedOfferOptions.csv"

write_csv_with_required_quotes(
    dynamic_price_option,
    dynamic_price_option_path,
    quoted_columns={"Provider", "OfferId", "UniqueOptionId"},
)

write_csv_with_required_quotes(
    dynamic_price_range,
    dynamic_price_range_path,
    quoted_columns={"Provider", "OfferId"},
)

write_csv_with_required_quotes(
    curated_offer_options,
    curated_offer_options_path,
    quoted_columns={
        "CurationProvider",
        "OfferId",
        "DealerId",
        "UniqueOptionId",
        "OptionId",
        "Eta",
        "ProductBrand",
        "DefeatPrimaryReason",
        "DefeatReasons",
    },
)

[dynamic_price_option_path, dynamic_price_range_path, curated_offer_options_path]

['DynamicPriceOption.csv', 'DynamicPriceRange.csv', 'CuratedOfferOptions.csv']

## Validation and Output Preview

The checks below confirm that:

- the three output tables have the required columns;
- `EnqueuedTimeSP` follows `DD/MM/YYYY` format;
- the generated CSV files exist;
- the raw CSV text shows the expected quote behavior.

In [28]:
assert list(dynamic_price_option.columns) == DYNAMIC_PRICE_OPTION_COLUMNS
assert list(dynamic_price_range.columns) == DYNAMIC_PRICE_RANGE_COLUMNS
assert list(curated_offer_options.columns) == CURATED_OFFER_OPTIONS_COLUMNS

for table in [dynamic_price_option, dynamic_price_range, curated_offer_options]:
    assert table["EnqueuedTimeSP"].str.match(r"^\d{2}/\d{2}/\d{4}$").all()

summary = pd.DataFrame(
    {
        "OutputFile": [
            dynamic_price_option_path,
            dynamic_price_range_path,
            curated_offer_options_path,
        ],
        "Rows": [
            len(dynamic_price_option),
            len(dynamic_price_range),
            len(curated_offer_options),
        ],
        "Columns": [
            len(dynamic_price_option.columns),
            len(dynamic_price_range.columns),
            len(curated_offer_options.columns),
        ],
    }
)

summary

,OutputFile,Rows,Columns
0,DynamicPriceOption.csv,56,5
1,DynamicPriceRange.csv,25,7
2,CuratedOfferOptions.csv,40,18


In [32]:
for path in [dynamic_price_option_path, dynamic_price_range_path, curated_offer_options_path]:
    print(f"\n{path}")
    print("-" * len(path))

    with open(path, "r", encoding="utf-8") as file:
        preview_lines = file.read().splitlines()[:3]

    print("\n".join(preview_lines))


DynamicPriceOption.csv
----------------------
Provider,OfferId,UniqueOptionId,BestPrice,EnqueuedTimeSP
"ApplyDynamicPricePerOption","56e0702c-0218-4626-8d3d-ae9d54b4503b","b0e296a9-0590-f0e0-8211-243a2ededb12",92.45,18/08/2021
"ApplyDynamicPricePerOption","56e0702c-0218-4626-8d3d-ae9d54b4503b","d6562c24-0b37-5fb4-8275-65b7b8b47b87",92.45,18/08/2021

DynamicPriceRange.csv
---------------------
Provider,OfferId,MinGlobal,MinRecommended,MaxRecommended,DifferenceMinRecommendMinTheory,EnqueuedTimeSP
"ApplyDynamicPriceRange","a6611d55-9624-4381-8cdd-323ee3689241",85.0,87.2,97.65,2.2,05/09/2021
"ApplyDynamicPriceRange","b8c636fa-8241-47dc-ac40-bdf438a04d9c",85.0,87.2,97.65,2.2,05/09/2021

CuratedOfferOptions.csv
-----------------------
CurationProvider,OfferId,DealerId,UniqueOptionId,OptionId,IsMobileDealer,IsOpen,Eta,ChamaScore,ProductBrand,IsWinner,MinimumPrice,MaximumPrice,DynamicPrice,FinalPrice,DefeatPrimaryReason,DefeatReasons,EnqueuedTimeSP
"ByPrice","149f0e53-ff85-425f-a01a-8710f0670